# Colab GPU benchmark
Select a GPU runtime and mount your own Google Drive. No Google passwords/tokens belong in code. Put licensed datasets under MyDrive/cardioagent/datasets/{PhysioNet,pascal}. The vault persists in Drive. No LLM fine-tuning is performed. Full five-model runs take substantially longer than smoke checks.

In [ ]:
from google.colab import drive
drive.mount("/content/drive")
from pathlib import Path
import os, subprocess
checkout=Path("/content/cardioagent-ops")
if not checkout.exists():
    subprocess.run(["git","clone","https://github.com/Taha2053/cadrioagent-ops.git",str(checkout)],check=True)
os.chdir(checkout)
subprocess.run(["pip","install","uv"],check=True)
subprocess.run(["uv","sync","--extra","deep-learning","--dev"],check=True)

In [ ]:
dataset=Path("/content/drive/MyDrive/cardioagent/datasets")
vault=Path("/content/drive/MyDrive/cardioagent/vault")
assert (dataset/"PhysioNet").is_dir() and (dataset/"pascal").is_dir(), "Upload both original datasets to Drive"
vault.mkdir(parents=True,exist_ok=True)
subprocess.run(["uv","run","--extra","deep-learning","python","-c","import tensorflow as tf; print(tf.config.list_physical_devices('GPU')); assert tf.config.list_physical_devices('GPU'), 'Select a GPU runtime'"],check=True)
subprocess.run(["uv","run","--extra","deep-learning","python","-m","training.benchmark","--dataset",str(dataset),"--vault",str(vault),"--tracking-uri",f"sqlite:///{vault}/mlflow.db","--rebuild-manifest","--epochs","20","--trials","6","--activate"],check=True)

In [ ]:
import zipfile
archive=Path("/content/cardioagent-results.zip")
with zipfile.ZipFile(archive,"w",zipfile.ZIP_DEFLATED) as output:
    for folder in ("models","runs","manifests"):
        for path in (vault/folder).rglob("*"):
            if path.is_file(): output.write(path,path.relative_to(vault))
from google.colab import files
files.download(str(archive))